In [ ]:
#Import libraries 
import sys
from pathlib import Path
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fasttext

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split


sys.path.append(str(Path.cwd().parent / "src"))
sys.path.append(str(Path.cwd().parent / "data"))
from helpers import pick_by_pos, write_fasttext, train_fasttext, train_svm, eval_metrics_ft, eval_metrics_svm ,normalize_doc_id

#Set random seeds to ensure reproducibility of results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#Experiment parameters
chunk_size = 2000
max_iteration = 30
TARGET_QUAL_AUTO = 2000
test_per_class = 20

# Dynamic confidence schedule
TAU_START = 0.70
TAU_END = 0.92
ramp_iteration = max_iteration

# Class imbalance controls
max_quant_to_qual_ratio= 1.5
max_quant_add_per_iteratioon= 200
undersample_quant_to_tual = 1.5

# Heuristic candidate review sample
heuristic_sample_frac = 0.03


fastText_configuration = {
        "name": "cfg",
        "epoch": 25,
        "lr": 0.35,
        "wordNgrams": 1,
        "dim": 100,
        "loss": "softmax",
        "minCount": 2,
        "minCountLabel": 0,
        "ws": 5,
        "thread": 4,
    }

best_metric = "f1_macro"
base_dir = Path.cwd().parent
data_dir = base_dir / "data"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)


In [6]:
df = pd.read_csv(data_dir/"data_with_categories.csv")

In [3]:
df

,Unnamed: 0,title,abstract,categories,doc_id,label
0,0,PolyMath: Evaluating Mathematical Reasoning in...,"In this paper, we introduce PolyMath, a multil...",cs.CL,1,cs.CL
1,1,"It's Not What Machines Can Learn, It's What We...",Can deep neural networks learn to solve any ...,cs.LG cs.NE stat.ML,2,cs.LG
2,2,"SOTorrent: Studying the Origin, Evolution, and...",Stack Overflow (SO) is the most popular ques...,cs.SE,3,cs.SE
3,3,A Survey on LLM-based Multi-Agent System: Rece...,LLM-based Multi-Agent Systems ( LLM-MAS ) ha...,cs.CL cs.MA,4,cs.CL
4,4,Comparing Reconstruction- and Contrastive-base...,Learning state representations enables robot...,cs.RO cs.LG,5,cs.RO
...,...,...,...,...,...,...
850300,850300,Identifiability and inference of non-parametri...,Mutation rate variation across loci is well ...,math.PR cs.CE cs.DS math.ST q-bio.PE stat.TH,850301,cs.CE
850301,850301,LSC-Eval: A General Framework to Evaluate Meth...,Lexical Semantic Change (LSC) provides insight...,cs.CL,850302,cs.CL
850302,850302,Big Ramsey degrees of 3-uniform hypergraphs,Given a countably infinite hypergraph $\math...,math.CO cs.DM math.LO,850303,cs.DM
850303,850303,Cross-lingual Approaches for the Detection of ...,"In this work, we present the first corpus fo...",cs.CL cs.LG,850304,cs.CL


In [11]:
# Split dataset into training and testing sets
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [31]:
#Define column names used for training
text_col = "abstract"
label_col = "label"
# Define output path 
out_path = output_dir / "train_ft_domain.txt"

In [36]:
write_fasttext(train_df, text_col, label_col, out_path)

In [37]:
test_outpath = output_dir / "test_ft_domain.txt"

In [38]:
write_fasttext(test_df, text_col, label_col, test_outpath)

In [23]:
model_outpath = output_dir / "domain_model.ftz"

In [27]:
model = train_fasttext(out_path, model_outpath, fastText_configuration)

Read 115M words
Number of words:  669996
Number of labels: 62
Progress: 100.0% words/sec/thread: 4609614 lr:  0.000000 avg.loss:  0.655713 ETA:   0h 0m 0s


In [39]:
# Lists to store true and predicted labels
y_true = []
y_pred = []

# Read test data formatted for FastText
with open(output_dir/"test_ft_domain.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # split label and text
        parts = line.split(" ", 1)
        if len(parts) < 2:
            continue
        true_label = parts[0].replace("__label__", "")
        text = parts[1]
        # predict
        pred_labels, _ = model.predict(text)
        pred_label = pred_labels[0].replace("__label__", "")
        
        y_true.append(true_label)
        y_pred.append(pred_label)

In [40]:
# Evaluation metrics
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Macro F1:", f1_score(y_true, y_pred, average="macro", zero_division=0))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.6885940927079107
Macro F1: 0.4123195878970101

Classification Report:
              precision    recall  f1-score   support

       cs.AI       0.39      0.33      0.36      8332
       cs.AR       0.57      0.51      0.54       966
       cs.CC       0.56      0.53      0.54      1674
       cs.CE       0.31      0.26      0.28      1353
       cs.CG       0.66      0.60      0.63      1188
       cs.CL       0.77      0.80      0.78     15261
       cs.CR       0.71      0.69      0.70      6381
       cs.CV       0.85      0.86      0.86     30055
       cs.CY       0.44      0.45      0.44      2892
       cs.DB       0.63      0.58      0.60      1473
       cs.DC       0.55      0.54      0.55      3433
       cs.DL       0.70      0.61      0.65       820
       cs.DM       0.52      0.52      0.52      1964
       cs.DS       0.62      0.66      0.64      3886
       cs.ET       0.46      0.38      0.42       769
       cs.FL       0.62      0.57      0.60       742